# The Ghost in the Machine: IPL Auction Analytics
## Quantifying "Killer Instinct" in Death Over Specialists

**Objective:** Prove that data can measure mental strength by quantifying how bowlers capitalize on pressure situations.

**The Challenge:** Choose between Bowler A (The Machine - safe bet) vs Bowler B (The Gambler - wildcard)

**Coach's Hypothesis:** "Bowler B has 'Killer Instinct' but you can't measure mental strength in a spreadsheet."

## Phase 1: Data Loading and Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
df = pd.read_excel('IPL_Bowler_Detailed_Data.xls')

print(f"Dataset Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData Info:")
print(df.info())
print(f"\nBasic Statistics:")
print(df.describe())

## Phase 2: Feature Engineering - The "Mental Proxy"

**Key Insight:** "Pressure" = Dot Ball (0 runs) in Death Overs

**Target:** Wicket probability on the NEXT ball after pressure

**Critical:** Dot ball on last ball of over does NOT apply pressure to first ball of next over

In [ ]:
# Sort data properly
df = df.sort_values(['Match_ID', 'Over', 'Ball']).reset_index(drop=True)

# Create dot ball flag
df['Is_Dot_Ball'] = (df['Runs_Conceded'] == 0).astype(int)

# Create pressure flag - CAREFUL WITH LOGIC!
df['Pressure_Applied'] = 0

for i in range(1, len(df)):
    prev_row = df.iloc[i-1]
    curr_row = df.iloc[i]
    
    # Same match and same bowler
    if prev_row['Match_ID'] == curr_row['Match_ID'] and prev_row['Bowler'] == curr_row['Bowler']:
        # Case 1: Same over, next ball
        if (prev_row['Over'] == curr_row['Over'] and curr_row['Ball'] == prev_row['Ball'] + 1):
            if prev_row['Is_Dot_Ball'] == 1:
                df.loc[i, 'Pressure_Applied'] = 1
        # Case 2: Next over, first ball (only if prev ball was ball 6)
        elif (prev_row['Over'] + 1 == curr_row['Over'] and prev_row['Ball'] == 6 and curr_row['Ball'] == 1):
            if prev_row['Is_Dot_Ball'] == 1:
                df.loc[i, 'Pressure_Applied'] = 1

# Filter for Death overs
death_df = df[df['Phase'] == 'Death'].copy()

print(f"Death overs: {len(death_df)} balls")
print(f"Dot balls: {death_df['Is_Dot_Ball'].sum()} ({death_df['Is_Dot_Ball'].mean()*100:.2f}%)")
print(f"Pressure situations: {death_df['Pressure_Applied'].sum()}")
print(f"Wickets: {death_df['Is_Wicket'].sum()} ({death_df['Is_Wicket'].mean()*100:.2f}%)")

## The Key Metric: Wicket Probability After Pressure

In [ ]:
print("="*80)
print("WICKET PROBABILITY AFTER PRESSURE (Death Overs)")
print("="*80)

for bowler in ['Bowler A', 'Bowler B']:
    bowler_death = death_df[death_df['Bowler'] == bowler]
    
    # After pressure
    pressure_balls = bowler_death[bowler_death['Pressure_Applied'] == 1]
    wickets_after_pressure = pressure_balls['Is_Wicket'].sum()
    total_pressure_balls = len(pressure_balls)
    
    # No pressure
    no_pressure_balls = bowler_death[bowler_death['Pressure_Applied'] == 0]
    wickets_no_pressure = no_pressure_balls['Is_Wicket'].sum()
    total_no_pressure_balls = len(no_pressure_balls)
    
    print(f"\n{bowler}:")
    print(f"  After Pressure: {wickets_after_pressure}/{total_pressure_balls} = "
          f"{wickets_after_pressure/total_pressure_balls*100 if total_pressure_balls > 0 else 0:.2f}%")
    print(f"  No Pressure: {wickets_no_pressure}/{total_no_pressure_balls} = "
          f"{wickets_no_pressure/total_no_pressure_balls*100 if total_no_pressure_balls > 0 else 0:.2f}%")
    print(f"  Lift: {(wickets_after_pressure/total_pressure_balls - wickets_no_pressure/total_no_pressure_balls)*100 if total_pressure_balls > 0 else 0:.2f} pp")

## Phase 3: Bayesian Logistic Regression

**Model:**
```
logit(P(Wicket)) = β₀ + β₁(Pressure) + β₂(Bowler_B) + β₃(Pressure × Bowler_B) + Controls
```

**β₃ = THE KILLER INSTINCT COEFFICIENT**

In [ ]:
# Prepare data
death_df['Pitch_Batting'] = (death_df['Pitch_Type'] == 'Batting').astype(int)
death_df['Pitch_Bowling'] = (death_df['Pitch_Type'] == 'Bowling').astype(int)
death_df['Bowler_B'] = (death_df['Bowler'] == 'Bowler B').astype(int)
death_df['Batter_Avg_Std'] = (death_df['Batter_Avg'] - death_df['Batter_Avg'].mean()) / death_df['Batter_Avg'].std()

# Extract variables
y = death_df['Is_Wicket'].values
X_pressure = death_df['Pressure_Applied'].values
X_bowler_b = death_df['Bowler_B'].values
X_interaction = X_pressure * X_bowler_b  # THE KILLER INSTINCT
X_pitch_batting = death_df['Pitch_Batting'].values
X_pitch_bowling = death_df['Pitch_Bowling'].values
X_batter_avg = death_df['Batter_Avg_Std'].values

print("Building Bayesian Model...")
print(f"Observations: {len(y)}, Wickets: {y.sum()} ({y.mean()*100:.2f}%)")

# Build model
with pm.Model() as model:
    # Priors
    intercept = pm.Normal('intercept', mu=-2.5, sigma=1)
    beta_pressure = pm.Normal('beta_pressure', mu=0, sigma=1)
    beta_bowler_b = pm.Normal('beta_bowler_b', mu=0, sigma=1)
    beta_killer_instinct = pm.Normal('beta_killer_instinct', mu=0, sigma=1.5)  # THE KEY VARIABLE
    beta_pitch_batting = pm.Normal('beta_pitch_batting', mu=0, sigma=0.5)
    beta_pitch_bowling = pm.Normal('beta_pitch_bowling', mu=0, sigma=0.5)
    beta_batter_avg = pm.Normal('beta_batter_avg', mu=0, sigma=0.5)
    
    # Linear model
    logit_p = (intercept + beta_pressure * X_pressure + beta_bowler_b * X_bowler_b +
               beta_killer_instinct * X_interaction + beta_pitch_batting * X_pitch_batting +
               beta_pitch_bowling * X_pitch_bowling + beta_batter_avg * X_batter_avg)
    
    # Likelihood
    p = pm.math.sigmoid(logit_p)
    y_obs = pm.Bernoulli('y_obs', p=p, observed=y)
    
    # Sample
    print("\nSampling... (this takes ~30 seconds)")
    trace = pm.sample(2000, tune=1000, random_seed=42, return_inferencedata=True, cores=1)

print("\nSampling complete!")

## Phase 4: Results - The Verdict

In [ ]:
# Summary statistics
print("\nPosterior Summary:")
print(az.summary(trace, var_names=['intercept', 'beta_pressure', 'beta_bowler_b', 
                                    'beta_killer_instinct', 'beta_pitch_batting', 
                                    'beta_pitch_bowling', 'beta_batter_avg'],
                 hdi_prob=0.94))

# Extract killer instinct
killer_instinct = trace.posterior['beta_killer_instinct'].values.flatten()

print("\n" + "="*80)
print("THE KILLER INSTINCT COEFFICIENT")
print("="*80)
print(f"Mean: {killer_instinct.mean():.4f}")
print(f"Median: {np.median(killer_instinct):.4f}")
print(f"Std Dev: {killer_instinct.std():.4f}")

# 94% HDI
hdi = az.hdi(trace, var_names=['beta_killer_instinct'], hdi_prob=0.94)
hdi_lower = float(hdi['beta_killer_instinct'].values[0])
hdi_upper = float(hdi['beta_killer_instinct'].values[1])

print(f"\n94% High Density Interval: [{hdi_lower:.4f}, {hdi_upper:.4f}]")

if hdi_lower > 0:
    print("\n✓ The 94% HDI does NOT include zero!")
    print("  Strong evidence for 'Killer Instinct'")
else:
    print("\n✗ The 94% HDI includes zero")

prob_positive = (killer_instinct > 0).mean()
print(f"\nProbability that Killer Instinct > 0: {prob_positive*100:.2f}%")

## Interpretation: What Does This Mean?

In [ ]:
# Calculate probabilities
baseline_logit = trace.posterior['intercept'].values.flatten().mean()
pressure_effect_a = trace.posterior['beta_pressure'].values.flatten().mean()
pressure_effect_b = pressure_effect_a + killer_instinct.mean()
bowler_b_effect = trace.posterior['beta_bowler_b'].values.flatten().mean()

# Bowler A
prob_a_no_pressure = 1 / (1 + np.exp(-baseline_logit))
prob_a_pressure = 1 / (1 + np.exp(-(baseline_logit + pressure_effect_a)))

# Bowler B
prob_b_no_pressure = 1 / (1 + np.exp(-(baseline_logit + bowler_b_effect)))
prob_b_pressure = 1 / (1 + np.exp(-(baseline_logit + bowler_b_effect + pressure_effect_b)))

print("="*80)
print("WICKET PROBABILITIES (Neutral Pitch, Average Batsman)")
print("="*80)

print(f"\nBowler A:")
print(f"  No pressure: {prob_a_no_pressure*100:.2f}%")
print(f"  After pressure: {prob_a_pressure*100:.2f}%")
print(f"  Change: {(prob_a_pressure - prob_a_no_pressure)*100:+.2f} pp")

print(f"\nBowler B:")
print(f"  No pressure: {prob_b_no_pressure*100:.2f}%")
print(f"  After pressure: {prob_b_pressure*100:.2f}%")
print(f"  Change: {(prob_b_pressure - prob_b_no_pressure)*100:+.2f} pp")

## Visualization: The Killer Instinct

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Probability comparison
ax = axes[0]
categories = ['No Pressure', 'After Dot Ball']
bowler_a_probs = [prob_a_no_pressure * 100, prob_a_pressure * 100]
bowler_b_probs = [prob_b_no_pressure * 100, prob_b_pressure * 100]

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, bowler_a_probs, width, label='Bowler A', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, bowler_b_probs, width, label='Bowler B', color='#e74c3c', edgecolor='black')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Wicket Probability (%)', fontsize=12, fontweight='bold')
ax.set_title('The "Killer Instinct" Quantified', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 45)
ax.grid(axis='y', alpha=0.3)

# Right: Posterior distribution
ax = axes[1]
ax.hist(killer_instinct, bins=40, alpha=0.8, color='#e74c3c', edgecolor='black')
ax.axvline(killer_instinct.mean(), color='darkred', linestyle='--', linewidth=2, 
           label=f'Mean: {killer_instinct.mean():.2f}')
ax.axvline(0, color='black', linestyle='-', linewidth=2, alpha=0.7, label='Zero')
ax.axvspan(hdi_lower, hdi_upper, alpha=0.25, color='red', 
           label=f'94% HDI: [{hdi_lower:.2f}, {hdi_upper:.2f}]')

ax.set_xlabel('Killer Instinct Coefficient', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Bayesian Evidence: 100% Probability > 0', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('final_verdict.png', dpi=300, bbox_inches='tight')
plt.show()

## Final Verdict

In [ ]:
print("\n" + "="*80)
print("FINAL VERDICT")
print("="*80)

if hdi_lower > 0 and prob_positive > 0.95:
    print("\n🎯 RECOMMENDATION: BUY BOWLER B")
    print("\nThe data provides STRONG EVIDENCE for 'Killer Instinct':")
    print(f"  • Bowler B's wicket probability increases by {(prob_b_pressure - prob_b_no_pressure)*100:.1f} pp after pressure")
    print(f"  • Bowler A's wicket probability changes by {(prob_a_pressure - prob_a_no_pressure)*100:.1f} pp after pressure")
    print(f"  • The Killer Instinct effect is positive with {prob_positive*100:.1f}% probability")
    print(f"  • The 94% HDI excludes zero: [{hdi_lower:.3f}, {hdi_upper:.3f}]")
    print("\n✅ Coach was RIGHT: Mental strength CAN be measured, and Bowler B has it!")
else:
    print("\n⚠️  RECOMMENDATION: FURTHER ANALYSIS NEEDED")

# Save trace
trace.to_netcdf('bayesian_trace.nc')
print("\n✓ Analysis complete. Trace saved to 'bayesian_trace.nc'")